In [16]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import joblib
import networkx as nx
import numpy as np
import pandas as pd
from pathlib import Path

In [17]:
CHECKPOINT_PATH = Path(r"C:\Users\KARTIK\OneDrive\Desktop\Master folder\Traffic flow project\checkpoints\traffic_service_bundle.joblib")

@dataclass
class ForecastResult:
    horizon_minutes: int
    predicted_density: float
    congestion_level: str
    status_color: str

In [18]:
class TrafficIntelligenceService:
    def __init__(self, bundle_path: Path = CHECKPOINT_PATH):
        if not bundle_path.exists():
            raise FileNotFoundError(f"Model bundle not found at {bundle_path}. Run src/export_pipeline.py first.")
        self.bundle = joblib.load(bundle_path)
        self.models = self.bundle["models"]
        self.feature_cols = self.bundle["feature_cols"]
        self.thresholds = self.bundle["congestion_thresholds"]
        self.road_network = self._initialize_road_network()

    def estimate_congestion(self, density: float) -> Tuple[str, str]:
        """Translates continuous queue density [0, 1] to Level of Service (LOS)."""
        if density < self.thresholds["free_flow"]:
            return "Free Flow (LOS A-B)", "green"
        elif density < self.thresholds["moderate"]:
            return "Moderate Traffic (LOS C-D)", "yellow"
        elif density < self.thresholds["congested"]:
            return "Heavy Congestion (LOS E)", "orange"
        else:
            return "Severe Gridlock (LOS F)", "red"

    def predict_horizons(self, current_features: pd.DataFrame) -> List[ForecastResult]:
        """Generates multi-step predictions and classifies congestion states."""
        # Align features
        x_input = current_features[self.feature_cols]
        results = []
        for h_idx, (key, model) in enumerate(self.models.items(), start=1):
            pred_density = float(model.predict(x_input)[0])
            pred_density = max(0.0, min(1.0, pred_density))  # Bound to physical limits [0, 1]
            level, color = self.estimate_congestion(pred_density)
            results.append(ForecastResult(
                horizon_minutes=h_idx * 15,
                predicted_density=round(pred_density, 4),
                congestion_level=level,
                status_color=color
            ))
        return results

    def _initialize_road_network(self) -> nx.DiGraph:
        """
        Creates a directed road graph surrounding the Delhi intersection:
        Nodes:
          - A: North Residential Gateway
          - B: Outer Arterial Junction
          - C: Central Intersection (Approach 1 Corridor)
          - D: Commercial Core / Central Business District
        """
        G = nx.DiGraph()
        # base_length in km, base_speed in km/h
        G.add_edge("A", "C", name="Direct Arterial via Approach 1", length_km=4.2, base_speed_kmh=50.0)
        G.add_edge("C", "D", name="Approach 1 Exit to Commercial Hub", length_km=2.1, base_speed_kmh=45.0)
        G.add_edge("A", "B", name="Ring Bypass Sector 1", length_km=5.5, base_speed_kmh=65.0)
        G.add_edge("B", "D", name="Ring Bypass Sector 2 to Hub", length_km=4.0, base_speed_kmh=60.0)
        return G

    def optimize_route(self, origin: str, destination: str, predicted_density: float) -> Dict:
        """
        Calculates dynamic travel time costs (Bureau of Public Roads formulation)
        and finds the optimal path using Dijkstra's algorithm.
        """
        G = self.road_network.copy()

        # Update dynamic cost on Approach 1 corridor based on predicted density
        for u, v, data in G.edges(data=True):
            length = data["length_km"]
            speed = data["base_speed_kmh"]

            # If the segment crosses the predicted Approach 1 bottleneck
            if "Approach 1" in data["name"]:
                # Speed drops as density increases: v = v_free * (1 - 0.75 * density)
                effective_speed = max(10.0, speed * (1.0 - 0.75 * predicted_density))
            else:
                effective_speed = speed * 0.90  # Normal bypass slowdown

            # Travel time in minutes
            travel_time_min = (length / effective_speed) * 60.0
            data["cost"] = travel_time_min

        # Compute shortest paths
        best_path = nx.shortest_path(G, source=origin, target=destination, weight="cost")
        best_cost = nx.shortest_path_length(G, source=origin, target=destination, weight="cost")

        # Compute alternate route
        all_paths = list(nx.all_simple_paths(G, source=origin, target=destination))
        alternate_paths = [p for p in all_paths if p != best_path]
        alt_summary = []
        for alt in alternate_paths:
            cost = sum(G[alt[i]][alt[i+1]]["cost"] for i in range(len(alt)-1))
            alt_summary.append({"path": " -> ".join(alt), "travel_time_min": round(cost, 2)})

        return {
            "origin": origin,
            "destination": destination,
            "recommended_path": " -> ".join(best_path),
            "estimated_travel_time_min": round(best_cost, 2),
            "alternatives": alt_summary,
            "approach1_density_used": round(predicted_density, 4)
        }

In [19]:
# Test Verification
service = TrafficIntelligenceService()

# Load a sample observation row from processed features
test_data = pd.read_parquet(Path(r"C:\Users\KARTIK\OneDrive\Desktop\Master folder\Traffic flow project\data\processed\delhi_features.parquet"))
sample_obs = test_data.iloc[-10:-9]  # Sample row

print("--- 1. Multi-Horizon Forecasts ---")
forecasts = service.predict_horizons(sample_obs)
for f in forecasts:
    print(f"t+{f.horizon_minutes}m: Density = {f.predicted_density} | {f.congestion_level}")

print("\n--- 2. Dynamic Route Optimization (Origin A -> Destination D) ---")
h1_density = forecasts[0].predicted_density
route_decision = service.optimize_route("A", "D", h1_density)
print(f"Recommended Route    : {route_decision['recommended_path']}")
print(f"Estimated Travel Time: {route_decision['estimated_travel_time_min']} mins")
for alt in route_decision["alternatives"]:
    print(f"Alternative Route    : {alt['path']} ({alt['travel_time_min']} mins)")

--- 1. Multi-Horizon Forecasts ---
t+15m: Density = 0.6172 | Heavy Congestion (LOS E)
t+30m: Density = 0.6194 | Heavy Congestion (LOS E)
t+45m: Density = 0.601 | Heavy Congestion (LOS E)
t+60m: Density = 0.5933 | Heavy Congestion (LOS E)

--- 2. Dynamic Route Optimization (Origin A -> Destination D) ---
Recommended Route    : A -> B -> D
Estimated Travel Time: 10.09 mins
Alternative Route    : A -> C -> D (14.6 mins)
